In [175]:
import os
import sys
import cv2
import numpy as np
from glob import glob

from botsort.bot_sort import BoTSORT


# --- Visualization Functions ---
def convert_boxes(box_data, to_format="tlbr"):
    """
    Converts a list of tuples (box, track_id) from 'tlwh' to 'tlbr' or ensures 'tlwh'.
    Input boxes are assumed to be in tlwh format: [x, y, w, h]
    """
    if not len(box_data):
        return []

    converted = []
    for box, track_id in box_data:
        x, y, w, h = box
        if to_format == "tlbr":
            converted.append(([x, y, x + w, y + h], track_id))
        elif to_format == "tlwh":
            converted.append(([x, y, w, h], track_id))
    return converted

def draw_boxes_on_bg(bg_image, box_data, current_format="tlwh"):
    box_color  = (255, 255, 255)
    text_color = (255, 255, 255)
    thickness  = 2

    for box, track_id in box_data:
        if current_format == "tlwh":
            x, y, w, h = map(int, box)
            pt1 = (x, y)
            pt2 = (x + w, y + h)
        elif current_format == "tlbr":
            x1, y1, x2, y2 = map(int, box)
            pt1 = (x1, y1)
            pt2 = (x2, y2)

        cv2.rectangle(bg_image, pt1, pt2, box_color, thickness)

        text   = f"GID: {track_id}"
        text_y = pt1[1] - 7 if pt1[1] - 7 > 15 else pt1[1] + 15
        cv2.putText(bg_image, text, (pt1[0], text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, thickness)

    return bg_image

# --- Path setup ---
path_to_botsort_parent = './'
if path_to_botsort_parent not in sys.path:
    sys.path.append(path_to_botsort_parent)

ROOT_FRAME_DIR = "/home/lab314/workspace/reid/ds_backend_reid/MCDPT/deepstream_npy_output"
# ROOT_FRAME_DIR = "/home/lab314/workspace/reid/ds_backend_reid/MCDPT/deepstream_npy_output"



In [176]:
from __future__ import annotations
import numpy as np
from collections import deque
from typing import Optional
import faiss

In [281]:

class global_tracklet:
    def __init__(
            self, 
            local_id: int,
            global_id: str,
            cam_source_born: int,
            feat: np.ndarray,
            bbox: np.ndarray,
            age: int
    ):
        self.global_id = global_id
        self.local_id = local_id
        self.cam_source_born = cam_source_born
        self.centroid = feat
        self.bbox = bbox
        self.age = 0

class global_registry:
    def __init__(
        self,
        match_threshold: float = 0.35,
        min_frames: int = 5,
        max_emb: int = 50,
        emb_dim: int = 256,
    ):
        self.match_threshold = match_threshold
        self.min_frames      = min_frames
        self.max_emb         = max_emb

        self._entries:       list[global_tracklet] = []
        self._tid_to_gid:    dict[int, int]     = {}   
        self._global_id_ctr: int                = 0

        self._emb_dim   = emb_dim        
        self._index_cpu = faiss.IndexFlatIP(emb_dim)   

        res             = faiss.StandardGpuResources()
        self._index     = faiss.index_cpu_to_gpu(res, 0, self._index_cpu)

        self._faiss_pos_to_gid: list[int] = []
        self._gid_to_entry: dict[int, global_tracklet] = {} 

    def step(self, 
             tracker, 
             frame_id: int,
             cam_index: int,
             ):
        
        # camera x
        current_tids = {t.track_id for t in tracker.tracked_stracks}

        print(f"Tracked Stracks len: {len(tracker.tracked_stracks)}")

        for track in tracker.tracked_stracks:
            tid = track.track_id
            feat = track.smooth_feat
            bbox = track.tlwh


            track.t_global_id = f"?_{cam_index}"
            current_cam = cam_index

            # if global id is already assigned to a tracklet, still need to use the bot_sort tracklet
            if track.t_global_id != 0 or not track.t_global_id.startswith("?"):
                gid = track.global_id
                entry = self._get_entry_by_gid(gid)
                if entry is not None and feat is not None:
                    entry.add_embedding(feat, frame_id, bbox)
            
            # if age is not much 
            if track.tracklet_len < self.min_frames:
                continue
            
            if feat is None:
                continue

            birth_cam = current_cam
            track.t_global_id = self._register_new(self, track.track_id, birth_cam, track.smooth_feat, track._tlwh, track.tracklet_len)
            self._global_id_ctr + 1

        

    def _get_entry_by_gid(self, gid: int) -> Optional[global_tracklet]:
            return self._gid_to_entry.get(gid)
    
    def _register_new(self, track_id, cam_index, feat, frame_id, bbox, age):
        gid = f"{self._global_id_ctr + 1}_{cam_index}"
        entry = global_tracklet(
            local_id=track_id, 
            global_id=gid,
            cam_source_born=cam_index,
            feat=feat,
            bbox = bbox,
            age = age
        )
        self._entries.append(entry)
        self._gid_to_entry[gid] = entry
        self._tid_to_gid[track_id] = gid

        # vec = entry.centroid.astype(np.float32).reshape(1, -1)
        # vec = np.ascontiguousarray(vec)
        # self._index.add(vec)                          
        # self._faiss_pos_to_gid.append(gid)            

        return gid


In [282]:
# self.tracked_stracks = []  
# self.lost_stracks = []  
# self.removed_stracks = []  


In [283]:
registry = global_registry(
    match_threshold=0.25,
    min_frames=5,
    max_emb=50,
    emb_dim=256,
)

_tracker_kwargs = dict(
    track_high_thresh=0.6,
    track_low_thresh=0.1,
    new_track_thresh=0.7,
    track_buffer=600,
    match_thresh=0.8,
    with_reid=True,
    proximity_thresh=0.7,
    appearance_thresh=0.25,
    euc_thresh=0.1,
    fuse_score=True,
    frame_rate=30,
    max_batch_size=8,
    map_len=None,
    real_data=True,
)

tracker1 = BoTSORT(**_tracker_kwargs)  # camera 0
tracker2 = BoTSORT(**_tracker_kwargs)  # camera 1

CAM1_TID_OFFSET = 0
CAM2_TID_OFFSET = 100000

cur_frame     = 0
ACTIVE_FORMAT = "tlwh"
frame_num = 0

In [284]:
for frame_num in range(1, 20):
    npy_path = f"{ROOT_FRAME_DIR}/batch_frame_{frame_num - 1}.npy"
    frame_content = np.load(npy_path, allow_pickle=True)


    detections1 = frame_content[0]['objects']
    for d in detections1:
        d['obj_meta'] = None

    detections2 = frame_content[1]['objects']

    for d in detections2:
        d['obj_meta'] = None

    tracker1.update(detections1)
    # tracker2.update(detections2)

    registry.step(tracker1, frame_num,cam_index=1)
    # registry.step(tracker1, frame_num,cam_index=2)

    print("\n[FN]", frame_num)

    print([track.t_global_id for track in tracker1.tracked_stracks])



Tracked Stracks len: 1

[FN] 1
['?_1']
Tracked Stracks len: 1

[FN] 2
['?_1']
Tracked Stracks len: 1

[FN] 3
['?_1']
Tracked Stracks len: 3

[FN] 4
['?_1', '?_1', '?_1']
Tracked Stracks len: 3

[FN] 5
['?_1', '?_1', '?_1']
Tracked Stracks len: 3

[FN] 6
['?_1', '?_1', '?_1']
Tracked Stracks len: 3

[FN] 7
['?_1', '?_1', '?_1']
Tracked Stracks len: 3

[FN] 8
['?_1', '?_1', '?_1']
Tracked Stracks len: 3

[FN] 9
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 10
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 11
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 12
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 13
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 14
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 15
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 16
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 17
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 18
['1_2', '1_3', '1_4']
Tracked Stracks len: 3

[FN] 19
['1_2', '1_3', '1_4']


In [93]:
registry.step(tracker=tracker1, frame_id=i, cam_index=1)

In [215]:
"?rgrghe"

True